# Capstone — ce que la sous-série PostTraining démontre

Cette sous-série a produit onze carnets. Chacun a entraîné quelque chose, mesuré quelque chose, et committé ses sorties. Le présent carnet ne réentraîne rien : il **lit les sorties committées**, en extrait les nombres, et calcule ce qu'ils autorisent à affirmer — et ce qu'ils n'autorisent pas.

C'est le geste que la sous-série ne pouvait pas faire sur elle-même, parce qu'il demande de tenir les onze protocoles côte à côte, et une chose désagréable : comparer chaque écart annoncé à la résolution de l'instrument qui l'a mesuré.

Il en sort une lecture unique en son genre dans la série : **aucune des quatre comparaisons de la sous-série ne survit à ses propres barres d'erreur**, et la raison est chaque fois la même — l'instrument mesure à une résolution plus grossière que l'écart qu'on lui demande de départager. Ce n'est pas un défaut des carnets : c'est la propriété des petits mondes qu'ils instruisent, et c'est précisément ce qu'un capstone doit rendre lisible.

Exécution CPU, sans GPU ni checkpoint : la matière première est déjà dans les fichiers committés.

## Ce que ce notebook suppose

Onze carnets, tous exécutés et committés avec leurs sorties. L'analyse en dépend entièrement : elle ne recalcule aucun entraînement, elle relit des preuves.

| Carnet | Ce qu'il établit | Noyau d'exécution |
|---|---|---|
| [RLPT-00](PostTraining/RLPT-00-Reward-Model-Bradley-Terry-Python.ipynb) | reward model Bradley-Terry écrit à la main, plafond de Bayes empirique, calibration | python3 (CPU) |
| [RLPT-00b](PostTraining/RLPT-00b-Biais-Dataset-Preferences-Python.ipynb) | biais de longueur dans un dataset de préférences, contrôle de longueur | python3 (CPU) |
| [RLPT-00c](PostTraining/RLPT-00c-Anatomie-Reward-Hacking-Python.ipynb) | anatomie du reward hacking, seuil analytique du rapport longueur/correction | coursia-ml-training (GPU) |
| [RLPT-00d](PostTraining/RLPT-00d-Reward-Trainer-TRL-Python.ipynb) | `RewardTrainer` de TRL contre la boucle maison, à budget égal | python3 (CPU) |
| [RLPT-00e](PostTraining/RLPT-00e-DPO-TRL-Python.ipynb) | DPO maison contre DPO TRL, même monde, même graines | python3 (CPU) |
| [RLPT-00f](PostTraining/RLPT-00f-GRPO-TRL-contre-PPO-Maison-Python.ipynb) | GRPO de TRL contre PPO maison, quatre graines appariées | python3 (CPU) |
| [RLPT-00g](PostTraining/RLPT-00g-PPO-TRL-Experimental-Python.ipynb) | PPO expérimental de TRL, fenêtre de versions et bras négatif | python3 (CPU) |
| [RLPT-01](PostTraining/RLPT-01-PPO-RLHF-Petit-Modele-Python.ipynb) | PPO/RLHF sur petit modèle, quatre graines, balayage de `beta` | python3 (CPU) |
| [RLPT-02](PostTraining/RLPT-02-GRPO-Minimal-Python.ipynb) | GRPO minimal sur LoRA, quatre graines | coursia-ml-training (GPU) |
| [RLPT-03](PostTraining/RLPT-03-Reward-Hacking-Python.ipynb) | contamination du few-shot et hacking mesuré sur un adaptateur réel | coursia-ml-training (GPU) |
| [RLPT-04](PostTraining/RLPT-04-DPO-Offline-contre-GRPO-Online-Python.ipynb) | DPO hors ligne contre GRPO en ligne, même budget, même graine | coursia-ml-training (GPU) |

Les quatre carnets GPU ne sont **pas** réexécutés ici : leur entraînement a déjà eu lieu, et le refaire produirait un autre run, pas une meilleure preuve. Ce que ce carnet ajoute est d'un autre ordre — l'arithmétique qui manque entre les lignes.

## 1. L'instrument : lire les sorties committées

Tout commence par une décision de méthode. Les nombres utilisés plus bas ne sont **pas recopiés** : ils sont **extraits** des sorties committées, par recherche de libellé. Si une sortie change, l'analyse suit — et si un libellé disparaît, l'extraction rend `None` au lieu de rendre un nombre inventé.

L'extraction est insensible aux accents, parce que les sorties du dépôt mélangent les deux orthographes selon le carnet.

In [1]:
import json
import math
import re
import unicodedata
from pathlib import Path

RACINE = None
for candidat in [Path.cwd(), *Path.cwd().parents]:
    if (candidat / "MyIA.AI.Notebooks").is_dir():
        RACINE = candidat / "MyIA.AI.Notebooks" / "RL"
        break
if RACINE is None:
    RACINE = Path("MyIA.AI.Notebooks/RL").resolve()
SOUS_SERIE = RACINE / "PostTraining"

CARNETS = [
    "RLPT-00-Reward-Model-Bradley-Terry-Python.ipynb",
    "RLPT-00b-Biais-Dataset-Preferences-Python.ipynb",
    "RLPT-00c-Anatomie-Reward-Hacking-Python.ipynb",
    "RLPT-00d-Reward-Trainer-TRL-Python.ipynb",
    "RLPT-00e-DPO-TRL-Python.ipynb",
    "RLPT-00f-GRPO-TRL-contre-PPO-Maison-Python.ipynb",
    "RLPT-00g-PPO-TRL-Experimental-Python.ipynb",
    "RLPT-01-PPO-RLHF-Petit-Modele-Python.ipynb",
    "RLPT-02-GRPO-Minimal-Python.ipynb",
    "RLPT-03-Reward-Hacking-Python.ipynb",
    "RLPT-04-DPO-Offline-contre-GRPO-Online-Python.ipynb",
]


def charger(nom):
    return json.loads((SOUS_SERIE / nom).read_text(encoding="utf-8"))


def sortie_texte(carnet):
    """Concatene les sorties texte de toutes les cellules de code."""
    morceaux = []
    for cellule in carnet["cells"]:
        if cellule.get("cell_type") != "code":
            continue
        for sortie in cellule.get("outputs", []):
            texte = sortie.get("text")
            if isinstance(texte, list):
                texte = "".join(texte)
            if texte:
                morceaux.append(texte)
    return "\n".join(morceaux)


def sans_accent(texte):
    decompose = unicodedata.normalize("NFD", texte)
    return "".join(c for c in decompose if unicodedata.category(c) != "Mn")


NOMBRE = re.compile(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?")


def nombres_apres(texte, libelle, combien=1):
    """Les `combien` premiers nombres qui suivent `libelle` (None si absent)."""
    plat = sans_accent(texte)
    ancre = plat.find(sans_accent(libelle))
    if ancre < 0:
        return [None] * combien
    debut = ancre + len(sans_accent(libelle))
    trouves = [float(m.group(0)) for m in NOMBRE.finditer(plat, debut)]
    return (trouves + [None] * combien)[:combien]


def nombre_apres(texte, libelle):
    return nombres_apres(texte, libelle, 1)[0]


TXT = {nom: sortie_texte(charger(nom)) for nom in CARNETS}
print(len(TXT), "carnets charges |", sum(len(v) for v in TXT.values()), "caracteres de sortie texte")


11 carnets charges | 59484 caracteres de sortie texte


In [2]:
lignes = []
non_executees = 0
for nom in CARNETS:
    carnet = charger(nom)
    code = [c for c in carnet["cells"] if c.get("cell_type") == "code"]
    executees = sum(1 for c in code if c.get("execution_count") is not None)
    sorties = sum(len(c.get("outputs", [])) for c in code)
    noyau = carnet.get("metadata", {}).get("kernelspec", {}).get("name", "?")
    court = "-".join(nom.split("-")[:2])
    non_executees += len(code) - executees
    lignes.append((court, noyau, len(code), executees, sorties))

print(f"{'carnet':<9} {'noyau':<21} {'code':>5} {'exec':>5} {'sorties':>8}")
print("-" * 52)
for court, noyau, total, executees, sorties in lignes:
    print(f"{court:<9} {noyau:<21} {total:>5} {executees:>5} {sorties:>8}")
print()
print("cellules de code sans execution_count :", non_executees)
assert non_executees == 0, "une sortie committée sans execution_count : la preuve est incomplète"
print("integrite OK : chaque cellule de code porte son execution_count et ses sorties.")


carnet    noyau                  code  exec  sorties
----------------------------------------------------
RLPT-00   python3                  15    15       21
RLPT-00b  python3                  13    13       14
RLPT-00c  coursia-ml-training      21    21       37
RLPT-00d  python3                  15    15       26
RLPT-00e  python3                  18    18       30
RLPT-00f  python3                  16    16       31
RLPT-00g  python3                  10    10       15
RLPT-01   python3                  14    14       14
RLPT-02   coursia-ml-training      11    11       28
RLPT-03   coursia-ml-training      12    12       38
RLPT-04   coursia-ml-training      12    12       62

cellules de code sans execution_count : 0
integrite OK : chaque cellule de code porte son execution_count et ses sorties.


### Lecture du tableau

Les onze carnets portent **zéro** cellule de code non exécutée. Ce n'est pas une formalité : c'est ce qui autorise tout ce qui suit. Une sortie committée sans `execution_count` serait une capture d'écran, pas une mesure — et l'analyse qui s'appuie dessus n'aurait aucune valeur de preuve.

Le tableau dit aussi autre chose, qui compte pour la suite : **sept carnets tournent sur CPU, quatre sur le noyau GPU `coursia-ml-training`**. Les comparaisons les plus coûteuses en calcul sont donc celles des carnets GPU, et ce sont elles qui disposent des plus petits jeux de test — nous y revenons en section 2.

## 2. Quatre comparaisons, et la résolution de leur instrument

La sous-série annonce plusieurs écarts. On les reprend un par un, avec la seule question qui tranche : **l'écart est-il plus grand que ce que l'instrument peut résoudre ?**

Quatre instruments différents sont à l'œuvre : des barres d'erreur inter-graines, un protocole apparié, un jeu de test de 48 items, et une exactitude qui ne bouge plus du tout.

In [3]:
def ecart_type(valeurs):
    n = len(valeurs)
    moyenne = sum(valeurs) / n
    return math.sqrt(sum((v - moyenne) ** 2 for v in valeurs) / (n - 1))


print("=== Comparaison 1 : RLPT-00d, RewardTrainer TRL contre la reference A.1 ===")
txt = TXT["RLPT-00d-Reward-Trainer-TRL-Python.ipynb"]
apparies = re.findall(r"seed\s+\d+\s*:\s*acc\s+([0-9.]+)\s*\(Bayes\s+([0-9.]+)\)", sans_accent(txt))
acc_trl = [float(a) for a, _ in apparies]
bayes_trl = [float(b) for _, b in apparies]
acc_ref, sd_ref = nombres_apres(txt, "reference A.1", 2)
moy_trl = sum(acc_trl) / len(acc_trl)
sd_trl = ecart_type(acc_trl)
z_deux_barres = (moy_trl - acc_ref) / math.sqrt(sd_trl ** 2 + sd_ref ** 2)
z_une_barre = (moy_trl - acc_ref) / sd_ref
print(f"  TRL           : {moy_trl:.4f} +- {sd_trl:.4f}  (n={len(acc_trl)} graines)")
print(f"  reference A.1 : {acc_ref:.4f} +- {sd_ref:.4f}")
print(f"  ecart         : {moy_trl - acc_ref:+.4f}")
print(f"  z sur la seule barre A.1 : {z_une_barre:+.2f}   <- le chiffre annonce par le carnet")
print(f"  z en combinant LES DEUX  : {z_deux_barres:+.2f}   <- le chiffre honnete")
print()

print("=== Comparaison 2 : RLPT-00f, GRPO TRL contre PPO maison, graines appariees ===")
txt = TXT["RLPT-00f-GRPO-TRL-contre-PPO-Maison-Python.ipynb"]
motif = r"seed\s+(\d+)\s*:\s*maison\s+([0-9.]+)->([0-9.]+)\s*\((\d+)s\)\s*\|\s*trl\s+([0-9.]+)->([0-9.]+)\s*\((\d+)s\)"
series = re.findall(motif, sans_accent(txt))
differences = [float(b) - float(f) for _, _, b, _, _, f, _ in series]
temps_maison = [int(t) for _, _, _, t, _, _, _ in series]
temps_trl = [int(t) for _, _, _, _, _, _, t in series]
moy_d = sum(differences) / len(differences)
sd_d = ecart_type(differences)
t_apparie = moy_d / (sd_d / math.sqrt(len(differences)))
print(f"  differences par graine (maison - trl) : {[round(d, 3) for d in differences]}")
print(f"  moyenne {moy_d:+.4f} | ecart-type {sd_d:.4f} | t apparie {t_apparie:+.2f}")
print(f"  soit, en combinant les deux bras : 2 sigma = {2 * sd_d:.4f}")
print(f"  temps moyen : maison {sum(temps_maison) / len(temps_maison):.0f}s | trl {sum(temps_trl) / len(temps_trl):.0f}s")
print()

print("=== Comparaison 3 : RLPT-04, la resolution du jeu de test ===")
txt = TXT["RLPT-04-DPO-Offline-contre-GRPO-Online-Python.ipynb"]
base = nombre_apres(txt, "BASELINE greedy")
dpo = nombre_apres(txt, "POST-DPO greedy")
grpo = nombre_apres(txt, "POST-GRPO greedy")
n_items = 48
un_exemple = 1 / n_items
print(f"  baseline {base:.3f} | DPO {dpo:.3f} | GRPO {grpo:.3f}  (n = {n_items} items)")
print(f"  1 item = {un_exemple:.4f} d'exactitude")
print(f"  DPO - baseline : {dpo - base:+.4f} = {(dpo - base) / un_exemple:.2f} item(s)")
print(f"  DPO - GRPO     : {dpo - grpo:+.4f} = {(dpo - grpo) / un_exemple:.2f} item(s)")
print(f"  erreur-type d'une proportion ~0.75 sur {n_items} items : {math.sqrt(0.75 * 0.25 / n_items):.4f}")
print()

print("=== Comparaison 4 : RLPT-02, une metrique qui ne bouge plus ===")
txt = TXT["RLPT-02-GRPO-Minimal-Python.ipynb"]
graines = re.findall(r"seed\s+\d+\s*:\s*reward moyen\s+([0-9.]+)\s*\|\s*accuracy greedy\s+([0-9.]+)", sans_accent(txt))
recompenses = [float(r) for r, _ in graines]
exactitudes = [float(a) for _, a in graines]
print(f"  exactitude greedy par graine : {exactitudes}  -> ecart-type {ecart_type(exactitudes):.4f}")
print(f"  recompense moyenne par graine: {[round(r, 3) for r in recompenses]}")
print(f"  ecart-type de la recompense  : {ecart_type(recompenses):.4f}")
print(f"  baseline greedy {nombre_apres(txt, 'Baseline greedy'):.2f} | post-RL greedy {nombre_apres(txt, 'Accuracy post-RL (greedy)'):.2f}")


=== Comparaison 1 : RLPT-00d, RewardTrainer TRL contre la reference A.1 ===
  TRL           : 0.6695 +- 0.0180  (n=4 graines)
  reference A.1 : 0.6580 +- 0.0080
  ecart         : +0.0115
  z sur la seule barre A.1 : +1.44   <- le chiffre annonce par le carnet
  z en combinant LES DEUX  : +0.58   <- le chiffre honnete

=== Comparaison 2 : RLPT-00f, GRPO TRL contre PPO maison, graines appariees ===
  differences par graine (maison - trl) : [0.152, 0.025, -0.066, 0.008]
  moyenne +0.0297 | ecart-type 0.0906 | t apparie +0.66
  soit, en combinant les deux bras : 2 sigma = 0.1811
  temps moyen : maison 121s | trl 80s

=== Comparaison 3 : RLPT-04, la resolution du jeu de test ===
  baseline 0.729 | DPO 0.792 | GRPO 0.771  (n = 48 items)
  1 item = 0.0208 d'exactitude
  DPO - baseline : +0.0630 = 3.02 item(s)
  DPO - GRPO     : +0.0210 = 1.01 item(s)
  erreur-type d'une proportion ~0.75 sur 48 items : 0.0625

=== Comparaison 4 : RLPT-02, une metrique qui ne bouge plus ===
  exactitude greedy 

### Lecture des quatre comparaisons

**1. `RLPT-00d` — le gain de 1,2 point est un demi-écart-type.** Le carnet rapporte `+0.012`, soigneusement exprimé en « 1,4 écarts-types A.1 » — mais ce dénominateur ne contient que la barre de la référence, pas celle du bras TRL. Combinées, les deux barres donnent `z = 0,58`. Un gain de 0,58 σ n'est pas un gain : c'est un tirage. Le carnet a raison de dire que les deux protocoles sont comparables ; il ne peut pas dire que l'un surpasse l'autre.

**2. `RLPT-00f` — l'écart apparié est plus petit que son propre bruit.** Les quatre différences par graine changent de signe (deux positives, une négative, une quasi nulle). Le `t` apparié vaut `0,66` : indistinguable de zéro. Le carnet conclut déjà `INCONCLUSIVE`, et le protocole apparié — plus puissant que la comparaison de moyennes qu'il utilise — ne le sauve pas.

**3. `RLPT-04` — 48 items, un item vaut 2,1 points.** C'est la phrase qui manquait. L'écart DPO–GRPO annoncé vaut **exactement un item**. L'écart DPO–baseline en vaut trois. Sur un jeu de 48 questions, l'erreur-type d'une proportion autour de 0,75 est de 6,25 points : toute conclusion qui porte sur moins de deux items est en dessous du plancher de l'instrument. Le carnet ne pouvait pas trancher DPO contre GRPO, et il ne le prétend pas — ce que la sous-série aurait pu dire, c'est **pourquoi**.

**4. `RLPT-02` — la métrique est saturée, donc elle ne mesure plus rien.** Quatre graines, exactitude `0,88` partout, écart-type `0,00`, et identique à la baseline. Ce n'est pas un échec de l'entraînement : c'est un succès de la baseline. Ce qui bouge, c'est la **récompense** — entre 0,44 et 0,67 selon la graine. Autrement dit, GRPO déplace bien quelque chose, mais pas la réponse finale. La conclusion honnête du carnet est donc double : le signal appris vit dans le canal de la récompense, et l'exactitude n'est pas l'instrument qui peut le voir.

Le motif commun est net : **quatre comparaisons, quatre instruments plus grossiers que l'écart mesuré**. Ce n'est pas une coquetterie statistique. C'est la raison pour laquelle la sous-série a construit des plafonds de Bayes explicites — c'est l'objet de la section suivante.

## 3. La part du plafond de Bayes

Quatre carnets établissent un **plafond de Bayes** : l'exactitude qu'atteindrait un juge optimal sur leur monde. C'est la seule échelle qui rende les exactitudes comparables entre carnets — et la seule qui dise combien de points il restait à gagner au moment où l'entraînement a commencé.

On calcule ici la part du plafond atteinte, et la marge résiduelle en points d'exactitude.

In [4]:
mesures = []

txt = TXT["RLPT-00-Reward-Model-Bradley-Terry-Python.ipynb"]
mesures.append(("RLPT-00", nombre_apres(txt, "vs accuracy"),
                nombre_apres(txt, "plafond de Bayes empirique E[max(p, 1-p)] (test)")))

txt = TXT["RLPT-00b-Biais-Dataset-Preferences-Python.ipynb"]
mesures.append(("RLPT-00b", nombre_apres(txt, "lenctrl, accuracy apres correction g*len"),
                nombre_apres(txt, "plafond de Bayes oracle")))

txt = TXT["RLPT-00d-Reward-Trainer-TRL-Python.ipynb"]
couples = re.findall(r"seed\s+\d+\s*:\s*acc\s+([0-9.]+)\s*\(Bayes\s+([0-9.]+)\)", sans_accent(txt))
mesures.append(("RLPT-00d", sum(float(a) for a, _ in couples) / len(couples),
                sum(float(b) for _, b in couples) / len(couples)))

txt = TXT["RLPT-00e-DPO-TRL-Python.ipynb"]
mesures.append(("RLPT-00e", nombre_apres(txt, "accuracy test"),
                nombre_apres(txt, "plafond d'accuracy (Bayes)")))

print(f"{'carnet':<10} {'accuracy':>9} {'plafond':>8} {'part':>7} {'marge (pts)':>12}")
print("-" * 50)
for nom, acc, plafond in mesures:
    part = acc / plafond
    print(f"{nom:<10} {acc:>9.3f} {plafond:>8.3f} {part:>6.1%} {100 * (plafond - acc):>11.1f}")
print()
print("Lecture : la marge est ce qu'il restait a gagner sur ce monde, pas ce qu'un bras a gagne.")


carnet      accuracy  plafond    part  marge (pts)
--------------------------------------------------
RLPT-00        0.657    0.702  93.6%         4.5
RLPT-00b       0.574    0.699  82.1%        12.5
RLPT-00d       0.669    0.696  96.2%         2.6
RLPT-00e       0.665    0.702  94.7%         3.7

Lecture : la marge est ce qu'il restait a gagner sur ce monde, pas ce qu'un bras a gagne.


### Lecture de la marge

Trois carnets sur quatre terminent entre **93,6 % et 96,2 %** de leur plafond. Il restait donc entre **2,6 et 4,5 points** d'exactitude à gagner sur ces mondes, et les carnets les ont pour l'essentiel pris.

C'est exactement ce qui rend les comparaisons de la section 2 non concluantes, et le lien est arithmétique : un bras qui vaut 96 % d'un plafond ne peut pas en gagner 5 de plus, donc tout écart annoncé dans cette zone est un écart **contre un plafond**, pas contre l'autre bras. `RLPT-00d` annonce 1,2 point dans une marge de 2,6 : le gain occupe près de la moitié de ce qui restait, avec un `z` de 0,58.

`RLPT-00b` est l'exception instructive : **82,1 %** seulement, soit **12,5 points** de marge encore ouverts. Le contrôle de longueur coûte cher — il retire du signal utile en même temps que du biais — et c'est le seul carnet de la sous-série dont le modèle garde une marge substantielle. Il ne s'en cache pas : le carnet présente ce chiffre comme le prix du contrôle, pas comme un échec.

Un plafond n'est pas une excuse. C'est un **dénominateur** : sans lui, « 0,657 d'exactitude » ne veut rien dire, et « +1,2 point » non plus.

## 4. La marge de reward hacking se calcule

`RLPT-00c` est le carnet le plus falsifiable de la sous-série, parce qu'il énonce un **seuil analytique** : le rapport entre le poids de la longueur et celui de la correction vaut `B / THETA`, soit `10 / 6`. Au-delà, l'argmax du proxy devient une réponse incorrecte — le modèle préfère la longueur à l'exactitude, et la politique optimise la mauvaise chose.

Ce seuil a une propriété précieuse : il se recalcule. On peut donc mesurer non pas « y a-t-il eu hacking ? » mais **de combien** le modèle appris a dépassé la frontière.

In [5]:
txt = TXT["RLPT-00c-Anatomie-Reward-Hacking-Python.ipynb"]

rapport_mesure = nombre_apres(txt, "rapport |wl / wc|")
budget, theta, seuil = nombres_apres(txt, "seuil analytique du rapport wl/wc", 3)
marge_relative = rapport_mesure / seuil - 1
print(f"  rapport mesure |wl/wc|   : {rapport_mesure:.4f}")
print(f"  seuil analytique B/THETA : {budget:.0f}/{theta:.0f} = {seuil:.4f}")
print(f"  marge au-dessus du seuil : {marge_relative:+.1%}")
print()

qualite_initiale, qualite_bras1 = nombres_apres(txt, "QUALITE VRAIE", 2)
qualite_bras2 = nombre_apres(txt, "qualite finale longueur-controle")
print(f"  qualite vraie : initiale {qualite_initiale:.4f} | bras 1 (biaise) {qualite_bras1:.4f} | bras 2 (longueur-controle) {qualite_bras2:.4f}")
print(f"  chute bras 1 : {qualite_bras1 - qualite_initiale:+.4f} | gain bras 2 : {qualite_bras2 - qualite_initiale:+.4f}")
print(f"  ecart bras 2 - bras 1 : {qualite_bras2 - qualite_bras1:+.4f}")
print()

canal_qualite = nombre_apres(txt, "canal qualite")
canal_longueur = nombre_apres(txt, "canal longueur")
proxy_somme = nombre_apres(txt, "proxy (somme)")
residu = canal_qualite + canal_longueur - proxy_somme
print("  decomposition de l'amelioration du proxy :")
print(f"    canal qualite  : {canal_qualite:+.4f}")
print(f"    canal longueur : {canal_longueur:+.4f}")
print(f"    somme          : {canal_qualite + canal_longueur:+.4f}")
print(f"    proxy annonce  : {proxy_somme:+.4f}")
print(f"    residu         : {residu:+.2e}   <- 0 = la decomposition se referme")
print()

bissection = re.search(r"\(largeur\s+([0-9.eE+-]+)\)", sans_accent(txt))
largeur_bissection = float(bissection.group(1)) if bissection else None
balayage = re.search(r"de largeur\s+([0-9.]+)", sans_accent(txt))
largeur_balayage = float(balayage.group(1)) if balayage else None
print(f"  encadrement du balayage  : largeur {largeur_balayage:.2f}")
print(f"  encadrement bissection   : largeur {largeur_bissection:.1e}")
if largeur_bissection and largeur_balayage:
    print(f"  rapport des deux         : {largeur_balayage / largeur_bissection:.2e} fois plus fin")

bornes = re.findall(r"wl/wc=([0-9.]+), seuil ([0-9.]+)", sans_accent(txt))
if len(bornes) >= 2:
    bas, seuil_bas = float(bornes[0][0]), float(bornes[0][1])
    haut, seuil_haut = float(bornes[1][0]), float(bornes[1][1])
    print()
    print("  verification du franchissement de seuil :")
    print(f"    borne basse {bas:.6f} < seuil {seuil_bas:.6f} : {bas < seuil_bas}")
    print(f"    borne haute {haut:.6f} > seuil {seuil_haut:.6f} : {haut > seuil_haut}")

  rapport mesure |wl/wc|   : 1.9923
  seuil analytique B/THETA : 10/6 = 1.6667
  marge au-dessus du seuil : +19.5%

  qualite vraie : initiale 0.4545 | bras 1 (biaise) 0.0170 | bras 2 (longueur-controle) 0.9902
  chute bras 1 : -0.4375 | gain bras 2 : +0.5357
  ecart bras 2 - bras 1 : +0.9732

  decomposition de l'amelioration du proxy :
    canal qualite  : -1.7006
    canal longueur : +3.7669
    somme          : +2.0663
    proxy annonce  : +2.0663
    residu         : +0.00e+00   <- 0 = la decomposition se referme

  encadrement du balayage  : largeur 0.50
  encadrement bissection   : largeur 3.0e-08
  rapport des deux         : 1.67e+07 fois plus fin

  verification du franchissement de seuil :
    borne basse 1.666261 < seuil 1.666667 : True
    borne haute 1.666937 > seuil 1.666667 : True


### Lecture de la marge

Le modèle appris dépasse le seuil analytique de **19,5 %**. Ce n'est pas un ordre de grandeur accidentel : c'est la conséquence directe du protocole d'annotation, où chaque paire est annotée avec un biais `a = 2,0`, et où le carnet montre que le rapport mesuré suit ce biais.

La décomposition par canal est le résultat le plus élégant de la sous-série, et elle **se vérifie** : `-1,7006 + 3,7669 = +2,0663`, exactement le gain de proxy annoncé, résidu nul à la précision d'affichage. Autrement dit, l'amélioration apparente du proxy se décompose intégralement en **une perte de qualité de 1,70** et **un gain de longueur de 3,77** — le second canal payant pour le premier et au-delà. Le bras biaisé ne « progresse » pas : il **convertit** de la qualité en longueur à un taux mesurable.

Le pendant est le bras 2 : contrôler la longueur rend `0,9902` de qualité vraie, soit plus que la politique initiale (`0,4545`) et au niveau de l'oracle (`0,9888`). Le même PPO, sur le même monde, apprend donc exactement ce qu'on lui donne à apprendre — ce qui est la définition opérationnelle d'un problème de spécification, et non d'optimisation.

## 5. Le coût par point de gain

Un écart non concluant n'est pas un écart sans valeur : il peut porter une différence de **coût**. C'est même le seul endroit où un harnais industriel peut gagner sur une boucle maison sans changer les résultats — moins de temps, moins de code à comprendre, moins de surface de bug.

On rapporte donc les deux : secondes par point de gain mesuré, et secondes par point de gain **crédible**.

In [6]:
txt = TXT["RLPT-00f-GRPO-TRL-contre-PPO-Maison-Python.ipynb"]
gain_maison, sd_maison, temps_maison_moyen = nombres_apres(txt, "gain maison", 3)
gain_trl, sd_trl, temps_trl_moyen = nombres_apres(txt, "gain trl", 3)
print("=== RLPT-00f : cout par point de gain ===")
print(f"  maison : gain {gain_maison:+.3f} +- {sd_maison:.3f} | {temps_maison_moyen:.0f}s | {temps_maison_moyen / gain_maison:.0f} s par point")
print(f"  trl    : gain {gain_trl:+.3f} +- {sd_trl:.3f} | {temps_trl_moyen:.0f}s | {temps_trl_moyen / gain_trl:.0f} s par point")
print(f"  rapport de cout par point : {(temps_maison_moyen / gain_maison) / (temps_trl_moyen / gain_trl):.2f}x en faveur de TRL")
print("  mais l'ecart de gain entre les bras est nul statistiquement (section 2).")
print("  -> le gain de TRL est un gain d'ingenierie, pas un gain de resultat.")
print()

txt = TXT["RLPT-00d-Reward-Trainer-TRL-Python.ipynb"]
rapport_annonce = nombre_apres(txt, "rapport de temps trl / maison")
temps_moyen_run = nombre_apres(txt, "temps moyen par run")
boucle_maison = re.search(r"en\s+([0-9.]+)\s*s", sans_accent(txt))
boucle_maison_s = float(boucle_maison.group(1)) if boucle_maison else None
print("=== RLPT-00d : deux denominateurs, deux rapports ===")
print(f"  rapport committe (boucle d'entrainement seule) : {rapport_annonce:.2f}x")
if boucle_maison_s:
    print(f"  boucle maison : {boucle_maison_s:.1f}s | run complet moyen : {temps_moyen_run:.1f}s")
    print(f"  rapport en incluant l'evaluation : {temps_moyen_run / boucle_maison_s:.2f}x")
print("  pour un ecart d'exactitude de +0.012, soit 0.58 sigma (section 2).")
print("  -> 39 secondes de plus par run pour une difference que l'instrument ne resout pas.")


=== RLPT-00f : cout par point de gain ===
  maison : gain +0.635 +- 0.100 | 121s | 191 s par point
  trl    : gain +0.605 +- 0.074 | 80s | 132 s par point
  rapport de cout par point : 1.44x en faveur de TRL
  mais l'ecart de gain entre les bras est nul statistiquement (section 2).
  -> le gain de TRL est un gain d'ingenierie, pas un gain de resultat.

=== RLPT-00d : deux denominateurs, deux rapports ===
  rapport committe (boucle d'entrainement seule) : 5.95x
  boucle maison : 6.9s | run complet moyen : 45.6s
  rapport en incluant l'evaluation : 6.61x
  pour un ecart d'exactitude de +0.012, soit 0.58 sigma (section 2).
  -> 39 secondes de plus par run pour une difference que l'instrument ne resout pas.


### Lecture du coût

`RLPT-00f` est le cas le plus net de la sous-série, et le plus utile : **TRL coûte 1,44 fois moins cher par point de gain, et son gain est statistiquement identique à celui de la boucle maison**. Les deux affirmations sont vraies en même temps, et c'est précisément ce qu'un ingénieur doit savoir. Le carnet a donc raison de conclure `INCONCLUSIVE` sur le résultat : la conclusion `INCONCLUSIVE` **est** le livrable, puisque ce qui reste mesurable est le coût.

`RLPT-00d` illustre un piège de lecture, sans que le carnet soit en faute. Le rapport committé de `5,95x` porte sur la **boucle d'entraînement** (`41,0 s` contre `6,9 s`) ; rapporté au **run complet** (`45,6 s`), le même rapport vaut `6,61x`. Les deux chiffres sont justes et désignent deux choses différentes. Un capstone sert à cela : nommer le dénominateur, parce qu'un rapport de temps sans son dénominateur est un nombre qui circule sans son sens.

La ligne la plus dure pour l'écosystème est ailleurs : `+0,012` d'exactitude pour 39 secondes de plus par run. Ce n'est pas un mauvais échange — c'est un **échange non mesuré**, et aucun des deux carnets ne prétend le contraire. La valeur de TRL dans cette sous-série est l'ergonomie (`1,8x` moins de lignes à écrire, `2` lignes pour sauver et recharger un checkpoint fidèle au bit près), pas la performance.

## 6. Ce que la sous-série ne démontre pas

Un capstone qui ne dirait que des succès ne serait pas un capstone. Ce que les onze carnets **ne** démontrent pas, mesuré ici :

- **Aucun transfert hors de leur monde.** Les quatre plafonds de Bayes sont *empiriques* : ils sont calculés sur le même générateur que l'entraînement. Un modèle à 98 % de son plafond sur un monde synthétique n'a rien prouvé sur un monde réel — il a prouvé qu'il apprend ce générateur. `RLPT-03` et `RLPT-04` vont plus loin, puisqu'ils travaillent sur un modèle de langage réel, mais leur jeu de test fait 48 items, ce qui plafonne leur résolution à 2,1 points.
- **Aucune comparaison inter-harnais concluante.** Les trois face-à-face TRL/maison (`00d`, `00e`, `00f`) rendent tous un écart non significatif. Ils établissent l'**équivalence** des implémentations, ce qui est un résultat, mais pas une supériorité.
- **Aucune borne sur la graine.** Les quatre graines de `00f` et `00d` sont les mêmes (`0, 1, 7, 42`) ; c'est un bon protocole, mais quatre graines ne donnent qu'un écart-type à trois degrés de liberté, donc des barres d'erreur elles-mêmes incertaines d'un facteur 2 environ. Les conclusions de non-significativité sont robustes ; les tailles d'effet ne le sont pas.
- **Aucune preuve que l'exactitude soit le bon objectif.** `RLPT-02` montre une exactitude figée à `0,88` pendant que la récompense varie de `0,44` à `0,67` : la métrique choisie peut être saturée alors que le modèle apprend encore. C'est le résultat le plus transférable de la sous-série vers un usage réel.

Ce dernier point est aussi la raison d'être de la section 2 : avant de comparer deux bras, il faut savoir **à quelle résolution** on mesure. C'est une question qu'on se pose une fois, et qui vaut pour tous les carnets suivants.

## Exercices

Les trois exercices reconstruisent les instruments de mesure du carnet. Ils portent sur les fonctions, pas sur les nombres : une fois écrites, elles doivent reproduire exactement les valeurs affichées plus haut.

**Exercice 1 — combiner deux barres d'erreur.**

Écrivez `z_combine(m1, s1, m2, s2)` qui rend l'écart réduit entre deux moyennes en combinant **les deux** écarts-types. Vérifiez qu'elle rend `+0,58` sur la comparaison 1, et non `+1,44`.

*Indice* : l'écart des moyennes se divise par la racine de la somme des carrés des deux écarts-types.

In [7]:
def z_combine(m1, s1, m2, s2):
    """Ecart reduit entre deux moyennes, en combinant LEURS DEUX barres."""
    result = None  # TODO etudiant
    return result


print("Exercice a completer : z_combine")
print("attendu sur la comparaison 1 : z = +0.58")


Exercice a completer : z_combine
attendu sur la comparaison 1 : z = +0.58


**Exercice 2 — la part du plafond.**

Écrivez `part_du_plafond(accuracy, plafond)` qui rend la fraction du plafond atteinte, et `marge_en_points(accuracy, plafond)` qui rend ce qu'il restait à gagner, en points d'exactitude. Les deux doivent rendre un résultat sur `RLPT-00b` (`0,574` et `0,699`) qui explique pourquoi ce carnet est le seul à garder une marge substantielle.

In [8]:
def part_du_plafond(accuracy, plafond):
    """Part du plafond de Bayes atteinte, entre 0 et 1."""
    result = None  # TODO etudiant
    return result


def marge_en_points(accuracy, plafond):
    """Points d'exactitude restant sous le plafond."""
    result = None  # TODO etudiant
    return result


print("Exercice a completer : part_du_plafond et marge_en_points")
print("attendu sur RLPT-00b : 82.1% et 12.5 points")


Exercice a completer : part_du_plafond et marge_en_points
attendu sur RLPT-00b : 82.1% et 12.5 points


**Exercice 3 — le t apparié.**

Écrivez `t_apparie(avant, apres)` qui rend la statistique `t` d'un protocole apparié, puis appliquez-la aux quatre graines de `RLPT-00f`. Vous devez retrouver `t = 0,66`.

Question de fond : pourquoi le `t` apparié est-il **plus puissant** que la comparaison des deux moyennes utilisée par le carnet, et pourquoi ne suffit-il pourtant pas à rendre l'écart significatif ici ?

In [9]:
def t_apparie(avant, apres):
    """Statistique t d'un protocole apparie (memes graines des deux cotes)."""
    print("Exercice a completer : t_apparie")
    return None


graines_maison = [0.977, 0.836, 0.839, 0.961]
graines_trl = [0.825, 0.811, 0.905, 0.953]
print("Exercice a completer : t_apparie")
print("attendu sur ces quatre graines : t = +0.66")


Exercice a completer : t_apparie
attendu sur ces quatre graines : t = +0.66


## À retenir

- **La résolution de l'instrument précède la comparaison.** Quatre comparaisons de la sous-série sont non concluantes, et quatre fois la raison est la même : l'écart est plus petit que ce que l'instrument peut départager. Un `z` se calcule avec **les deux** barres d'erreur, pas avec celle de la référence seule — `1,44 σ` devient `0,58 σ` dès qu'on le fait correctement.
- **Un jeu de test de 48 items résout 2,1 points.** Un écart d'un item est du bruit ; deux items commencent à être une mesure. Avant d'interpréter un écart d'exactitude, divisez-le par `1 / n`.
- **Le plafond de Bayes est un dénominateur, pas une excuse.** Trois carnets terminent entre 93,6 % et 96,2 % de leur plafond : il n'y avait presque rien à gagner, et les écarts annoncés dans cette zone sont des écarts contre le plafond. Le seul carnet qui garde une marge (`RLPT-00b`, 82,1 %, soit 12,5 points) la paie au prix du contrôle de longueur.
- **Le reward hacking se mesure.** `RLPT-00c` dépasse son seuil analytique de 19,5 %, et sa décomposition par canal se referme exactement : `-1,70` de qualité pour `+3,77` de longueur, soit `+2,07` de proxy. Le modèle ne progresse pas, il convertit.
- **Une métrique d'exactitude peut être saturée alors que le modèle apprend.** `RLPT-02` le montre : exactitude figée à `0,88` sur quatre graines, récompense qui varie de `0,44` à `0,67`. Choisir la métrique fait partie de l'expérience.
- **Un écart non concluant n'est pas un écart sans valeur.** Il laisse place au coût : `TRL` coûte 1,44 fois moins cher par point, à résultat statistiquement identique. C'est un gain d'ingénierie, et le dire ainsi est plus honnête que de le vendre comme une performance.

## Vérifiez votre compréhension

1. Le carnet `RLPT-00d` annonce `+0,012` d'exactitude et l'exprime en « 1,4 écarts-types A.1 ». Pourquoi ce dénominateur est-il le mauvais, et quel chiffre obtient-on en combinant les deux barres ?
2. Un carnet rapporte `+0,021` d'exactitude sur un jeu de 48 items. Combien d'items cela représente-t-il, et que peut-on en conclure ?
3. Un modèle atteint `0,688` pour un plafond de Bayes de `0,702`. Combien de points restait-il à gagner, et pourquoi un gain annoncé de 2 points dans cette zone est-il suspect ?
4. `RLPT-00c` trouve `|wl/wc| = 1,9923` pour un seuil analytique de `10/6`. Le modèle a-t-il dépassé le seuil, et de combien ?
5. Pourquoi `-1,7006 + 3,7669 = +2,0663` est-il un résultat plus fort qu'une simple liste de trois nombres ?
6. `RLPT-02` obtient `0,88` d'exactitude avec un écart-type inter-graines de `0,00`, et une récompense qui varie de `0,44` à `0,67`. Que mesure réellement l'entraînement, et quelle métrique faudrait-il rapporter ?
7. Pourquoi le `t` apparié de l'exercice 3 est-il plus puissant que la comparaison de deux moyennes indépendantes, et pourquoi ne suffit-il pas ici ?
8. Le rapport de temps de `RLPT-00d` vaut `5,95x` ou `6,61x` selon le dénominateur. Quel est le bon chiffre, et que faut-il toujours écrire à côté d'un rapport de temps ?

## Pour aller plus loin

- **Le plafond de Bayes comme dénominateur** : `RLPT-00` le construit à la main en énumérant les paires et en calculant `E[max(p, 1-p)]` ; c'est la brique qui rend tout le reste interprétable, et elle se réimplémente en une vingtaine de lignes.
- **La décomposition par canal** : `RLPT-00c` sépare le proxy en qualité et longueur, puis vérifie que la somme retombe sur le gain mesuré. C'est le patron à réutiliser dès qu'une récompense est une somme de termes dont un seul est l'objectif.
- **Les protocoles appariés** : `RLPT-00f` fournit quatre graines appariées — la matière première d'un test apparié — mais ne l'exploite pas. L'exercice 3 comble ce manque.
- **La fidélité des checkpoints** : `RLPT-00d` vérifie qu'un modèle rechargé produit exactement les mêmes poids (`0,0e+00` d'écart) et des logits à `2,91e-03`, entièrement expliqués par l'arrondi `bfloat16`. Une différence de logits n'est pas toujours un bug.
- **La fenêtre de disponibilité d'un outil** : `RLPT-00g` documente que `trl.experimental.ppo` a existé de `trl 1.10.0` à `1.12.0` puis a été retiré en `1.13.0`. Un bras négatif documenté vaut mieux qu'un bras absent.
- **Le retour au socle** : le parcours principal de la série reprend au [README de la série](../README.md), et l'arc de la sous-série est décrit dans son [README](PostTraining/README.md).